# CIC-IDS-2017 - Data Cleaning

This notebook iterates through all CSV files in the TrafficLabelling directory, applies cleaning steps, and saves the cleaned files.

In [25]:
import pandas as pd
import numpy as np
import os
import dotenv
import warnings

warnings.filterwarnings('ignore')
dotenv.load_dotenv()

True

In [26]:
data_dir = os.getenv('DATA_DIR', './data')
traffic_dir = os.path.join(data_dir, 'GeneratedLabelledFlows', 'TrafficLabelling')

csv_files = sorted([f for f in os.listdir(traffic_dir) if f.endswith('.csv') and '_cleaned' not in f])
print(f'Found {len(csv_files)} CSV files in {traffic_dir}:\n')
for f in csv_files:
    print(f'  {f}')

Found 8 CSV files in /mnt/data/capstone/GeneratedLabelledFlows/TrafficLabelling:

  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Friday-WorkingHours-Morning.pcap_ISCX.csv
  Monday-WorkingHours.pcap_ISCX.csv
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Tuesday-WorkingHours.pcap_ISCX.csv
  Wednesday-workingHours.pcap_ISCX.csv


In [27]:
id_cols = ['Flow ID', 'Timestamp']

correlated_drops = [
    'Total Backward Packets',
    'Subflow Fwd Packets',
    'Subflow Bwd Packets',
    'Total Length of Bwd Packets',
    'Subflow Bwd Bytes',
    'Subflow Fwd Bytes',
    'Avg Fwd Segment Size',
    'Avg Bwd Segment Size',
    'Fwd Header Length.1',
    'Fwd PSH Flags',
    'CWE Flag Count',
    'ECE Flag Count',
    'Average Packet Size',
    'Fwd IAT Total',
    'Fwd IAT Max',
    'Idle Max',
    'Idle Min',
]

highly_skewed_cols = [
    'Fwd Header Length',
    'Total Length of Fwd Packets',
    'Bwd Header Length',
    'min_seg_size_forward',
    'act_data_pkt_fwd',
    'Total Fwd Packets',
    'Fwd URG Flags',
    'RST Flag Count',
    'Active Min',
    'Flow Bytes/s',
    'Active Std',
    'Active Mean',
    'Active Max',
    'Flow IAT Min',
    'Bwd Packets/s',
    'Fwd Packet Length Min',
    'Down/Up Ratio',
    'Fwd Packet Length Std',
    'Idle Std',
    'Min Packet Length',
    'Bwd IAT Min',
    'Fwd Packet Length Max',
    'Fwd IAT Min',
    'Fwd Packet Length Mean',
    'Flow IAT Mean',
    'Bwd IAT Mean',
    'Fwd IAT Mean',
    'Bwd IAT Std',
    'Fwd Packets/s',
    'Flow Packets/s',
    'Packet Length Variance',
    'Init_Win_bytes_backward',
    'FIN Flag Count',
    'Bwd Packet Length Min',
    'Bwd IAT Max',
    'SYN Flag Count',
    'Flow IAT Std',
    'Bwd Packet Length Std',
    'Fwd IAT Std',
    'Idle Mean',
    'Bwd Packet Length Max',
    'Flow IAT Max',
    'Bwd IAT Total',
    'Packet Length Std',
    'Max Packet Length',
    'URG Flag Count',
    'Init_Win_bytes_forward',
    'Bwd Packet Length Mean',
    'Packet Length Mean',
    'Flow Duration',
    'Destination Port',
]

def clean(df):
    rows_before = len(df)
    non_id_cols = [c for c in df.columns if c not in id_cols]
    df = df.dropna(subset=non_id_cols, how='all')
    rows_after = len(df)
    print(f'  Removed {rows_before - rows_after:,} entirely-null rows (ignoring ID columns)')

    cols_before = df.shape[1]
    drops = [c for c in correlated_drops if c in df.columns]
    df = df.drop(columns=drops)
    print(f'  Dropped {len(drops)} near-perfectly correlated columns ({cols_before} -> {df.shape[1]})')

    rate_cols = ['Flow Bytes/s', 'Flow Packets/s']
    for col in rate_cols:
        if col in df.columns:
            n_fixed = df[col].isna().sum() + np.isinf(df[col]).sum()
            df[col] = df[col].replace([np.inf, -np.inf], 0).fillna(0)
            print(f'  Replaced {n_fixed:,} null/infinite values with 0 in {col}')

    skew_cols = [c for c in highly_skewed_cols if c in df.columns]
    for col in skew_cols:
        df[col] = np.log1p(np.maximum(df[col], 0))
    print(f'  Applied log1p transformation to {len(skew_cols)} highly skewed columns')

    return df

In [28]:
all_cleaned = []
for filename in csv_files:
    filepath = os.path.join(traffic_dir, filename)
    df = pd.read_csv(filepath, encoding='cp1252')
    df.columns = df.columns.str.strip()
    print(f'{filename}: {df.shape[0]:,} rows')

    df = clean(df)
    all_cleaned.append(df)

    out_name = filename.replace('.csv', '_cleaned.csv')
    out_path = os.path.join(traffic_dir, out_name)
    df.to_csv(out_path, index=False, float_format='%.6f')
    print(f'  Saved {df.shape[0]:,} rows to {out_name}\n')

combined = pd.concat(all_cleaned, ignore_index=True)
combined_path = os.path.join(traffic_dir, 'all_cleaned.csv')
combined.to_csv(combined_path, index=False, float_format='%.6f')
print(f'Combined all files: {combined.shape[0]:,} rows, {combined.shape[1]} columns')
print(f'Saved to {combined_path}')

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows
  Removed 0 entirely-null rows (ignoring ID columns)
  Dropped 17 near-perfectly correlated columns (85 -> 68)
  Replaced 34 null/infinite values with 0 in Flow Bytes/s
  Replaced 34 null/infinite values with 0 in Flow Packets/s
  Applied log1p transformation to 51 highly skewed columns
  Saved 225,745 rows to Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.csv

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows
  Removed 0 entirely-null rows (ignoring ID columns)
  Dropped 17 near-perfectly correlated columns (85 -> 68)
  Replaced 371 null/infinite values with 0 in Flow Bytes/s
  Replaced 371 null/infinite values with 0 in Flow Packets/s
  Applied log1p transformation to 51 highly skewed columns
  Saved 286,467 rows to Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.csv

Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows
  Removed 0 entirely-null rows (ignoring ID columns)
  Dropped 17 ne